# 02 — Interview Generation

**Task.** Given a candidate's first four interview question/response pairs, generate their fifth response. The score is mean cosine similarity (using SentenceTransformer all-MiniLM-L6-v2) between the generated response and the actual fifth response.

**What the winners got.** Hungry Llama .512 · Akben .496 · Wonderlic .460 · PAID .440.

**The pattern.** This is the only task where PAID — the overall winner — finished last. The reason is that they treated the task as "what's a plausible answer to this question?" when the metric actually rewards "what would *this candidate* say?" Style preservation matters more than content quality.

## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
from src.adapters import InterviewAdapter
from src.harness import Harness, CallSpec
from src.scoring import avg_cosine_similarity
from src.run import load_csv, _load_labels, DATA_DIR

print(f"Data directory: {DATA_DIR.resolve()}")
print(f"Interview inputs (train): {(DATA_DIR / 'interview_train.csv').exists()}")
print(f"Interview inputs (dev):   {(DATA_DIR / 'interview_val_public.csv').exists() or (DATA_DIR / 'interview_dev_inputs.csv').exists()}")
print(f"Interview inputs (test):  {(DATA_DIR / 'interview_test_public.csv').exists() or (DATA_DIR / 'interview_test_inputs.csv').exists()}")

## The data

Each row in the interview dataset is one candidate's full set of five Q/R pairs. Train rows have all five; dev/test rows have Q1-Q5 and R1-R4 (the model needs to generate R5). The questions are standardized (every candidate answers the same Q1, the same Q2, etc.), so the only thing distinguishing one candidate from another is their *prior responses*.

Row IDs are Qualtrics-style strings like `R_1SONLQw1Nbpff7r`, since the data came from a Qualtrics survey.

Below is what a single test row looks like. The questions are the same across candidates; only the responses (R1-R4) differentiate them.

In [ ]:
interview_test = load_csv(DATA_DIR / "interview_test_public.csv")
if interview_test:
    print(f"Loaded {len(interview_test)} test interview rows")
    sample = interview_test[0]
    print(f"\nColumns: {list(sample.keys())}")
    print(f"\nSample test row (id={sample.get('_id', '?')}):")
    for k in ["Q1", "R1", "Q2", "R2", "Q3", "R3", "Q4", "R4", "Q5"]:
        if k in sample:
            print(f"  {k}: {sample[k][:120]}...")
else:
    print("Interview test inputs not present.")
    print("Synthetic example for prompt inspection:")
    interview_test = [{
        "_id": "R_synthetic",
        "Q1": "Tell me about yourself.",
        "R1": "I'm a recent grad with a background in marketing. I love working on creative campaigns and have hands-on experience from an internship at a startup.",
        "Q2": "What's your biggest professional achievement so far?",
        "R2": "Probably leading a social media push during my internship — we tripled engagement in two months by leaning into video content.",
        "Q3": "How do you handle stress?",
        "R3": "I lean on lists and short walks. Breaking things into smaller pieces helps me a lot.",
        "Q4": "What's a weakness you're working on?",
        "R4": "Sometimes I over-prepare for meetings to the point of slowing things down. I'm working on speaking up sooner with rougher ideas.",
        "Q5": "Where do you see yourself in five years?",
    }]

## PAID Team approach — "question-centered"

PAID Team scored last on this task with .440. Their reported approach:

1. Collect all responses to Q5 across train/val/test sets.
2. Use them as few-shot examples (with their corresponding Q1-Q4/R1-R4 context).
3. Ask GPT-4 to generate a response for the test row's Q5.

The flaw: pooling across candidates dilutes the per-candidate style signal. The model learns "here's what Q5 answers tend to look like" and produces a generic but plausible answer. Generic-vs-generic cosine is okay; generic-vs-specific cosine is bad.

We don't reconstruct this in detail — it's mostly a cautionary example.

## Hungry Llama approach — Big-5 personality conditioning

Hungry Llama scored .512, best in class. Their approach:

1. For each candidate, infer Big-5 personality scores from R1-R4 using BART zero-shot classification (with candidate labels "openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism").
2. Logit-transform the probabilities, take the standard deviation as a "confidence" feature.
3. Compute average sentence count and reading level across R1-R4.
4. Inject these as conditioning into the GPT-4 generation prompt: "Write at the reading level of a high-school graduate. Match a personality high in conscientiousness and extraversion. Average response length: 3 sentences."

The reading-level and sentence-count signals are cheap to compute and probably worth as much as the personality scores. The personality scores add discrimination on tonal dimensions (an introvert candidate has reserved-sounding prior responses; conditioning the model preserves that).

In [ ]:
# Reconstruction of the personality-conditioning preprocessing step.
# In a real run, this would be cached and joined with the test rows before generation.

def infer_style_features(prior_responses):
    '''Lightweight version of Hungry Llama's approach. Returns a dict of
    style features that can be injected into the generation prompt.
    Real implementation would use BART zero-shot via huggingface; here
    we just sketch the surface metrics.
    '''
    import re
    text = " ".join(prior_responses)
    
    # Sentence and word count
    sentences = re.split(r'[.!?]+', text)
    sentences = [s for s in sentences if s.strip()]
    avg_sent_len = (len(text.split()) / len(sentences)) if sentences else 10
    avg_response_sent_count = len(sentences) / len(prior_responses)
    
    # Approximate reading level via word complexity proxy
    # (real implementation: textstat.flesch_kincaid_grade)
    long_word_ratio = sum(1 for w in text.split() if len(w) >= 8) / max(1, len(text.split()))
    if long_word_ratio < 0.05:
        reading_level = "middle school"
    elif long_word_ratio < 0.10:
        reading_level = "high school"
    elif long_word_ratio < 0.15:
        reading_level = "college graduate"
    else:
        reading_level = "professional"
    
    return {
        "avg_words_per_sentence": round(avg_sent_len, 1),
        "avg_sentences_per_response": round(avg_response_sent_count, 1),
        "reading_level": reading_level,
    }


sample_row = interview_test[0]
prior_responses = [sample_row[f"R{i}"] for i in range(1, 5)]
features = infer_style_features(prior_responses)
print(f"Style features for {sample_row.get('_id', 'sample')}:")
for k, v in features.items():
    print(f"  {k}: {v}")

## Akben approach — generate N, pick by cosine to input

Akben scored .496, second on this task. Their approach:

1. Ask GPT-4 to generate N candidate responses (with high temperature for diversity).
2. Compute cosine similarity between each candidate and the *concatenation of R1-R4*.
3. Return the candidate with the highest similarity to the prior responses.

The intuition: the metric is similarity to the actual R5, which by assumption matches the candidate's style. So picking the candidate completion that's most similar to R1-R4 should preserve style. It's a self-consistency variant where instead of voting for the most common answer, you vote for the most stylistically continuous answer.

This is clever and cheap. We implement it below.

In [ ]:
def generate_akben_style(row, harness, model="gpt-4o-2024-08-06", n_candidates=5):
    '''Generate N candidates, return the one closest to the candidate's prior responses.'''
    from src.adapters import InterviewAdapter
    from sentence_transformers import SentenceTransformer
    import numpy as np
    
    adapter = InterviewAdapter()
    base_messages = adapter.build_messages(row, examples=[])
    
    # Generate N candidates with varying temperature
    candidates = []
    for i in range(n_candidates):
        spec = CallSpec(
            messages=base_messages,
            model=model,
            temperature=0.9,
            seed=1234 + i,
        )
        # text = harness.call(spec)
        # candidates.append(adapter.parse(text, row))
        candidates.append(f"<candidate {i+1} text>")  # placeholder
    
    # Score each candidate by cosine to the concatenated prior responses
    # embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    # prior = " ".join(row[f"R{i}"] for i in range(1, 5))
    # prior_emb = embedder.encode([prior], normalize_embeddings=True)[0]
    # cand_embs = embedder.encode(candidates, normalize_embeddings=True)
    # sims = cand_embs @ prior_emb
    # best_idx = int(np.argmax(sims))
    # return candidates[best_idx]
    
    print(f"Would generate {n_candidates} candidates, embed them, return the one closest to R1-R4.")
    return candidates[0]

generate_akben_style(sample_row, None, n_candidates=5)

## The unified-harness approach

The InterviewAdapter takes a lighter approach: a system prompt that instructs the model to match the style and personality of the prior responses, with a 120-word cap. No external few-shot (since the prior responses *are* the context). Optional Akben-style reranking via the harness's self-consistency wrapper.

We deliberately don't replicate Hungry Llama's full personality pipeline. The reason: with 2026 models, the instruction "match the style, tone, vocabulary, and personality reflected in your previous responses" reliably gets the model to do it. The BART personality classification was a useful explicit signal in 2024; today it's mostly redundant with what a good system prompt does for free.

If you want the personality features anyway, the cell above computes them; just inject the resulting dict into the system prompt before calling `adapter.build_messages`.

In [ ]:
adapter = InterviewAdapter()
print(f"Adapter: {adapter.task_name}")
print(f"Max words: {adapter.max_words}")
print(f"Response format: free text (no structured output)")
print()

sample_messages = adapter.build_messages(sample_row, examples=[])
print(f"Number of messages: {len(sample_messages)}")
print(f"\n--- System prompt ---\n{sample_messages[0]['content']}")
print(f"\n--- User turn (truncated) ---\n{sample_messages[1]['content'][:400]}...")

## End-to-end run

In [ ]:
from src.run import run_task

result = run_task(
    task="interview",
    split="dev",
    model="gpt-4o-2024-08-06",
    self_consistency=1,
    output_path=None,
    row_id=None,
    similarity_examples=False,  # interview has no external few-shot
)
if result["status"] == "ok":
    print(f"Interview dev: n={result['n']}, avg_cosine={result.get('score', 'n/a'):.4f}")
else:
    print(f"Status: {result['status']}")
    print(f"Message: {result.get('message')}")

## Discussion

The interview task is the most metric-dependent of the four. Different generation approaches produce different stylistic distributions, and which one wins on cosine similarity is partly a property of the metric (not of "quality").

What generalizes:
- **Preserve the per-instance signal.** Whatever distinguishes this candidate from others is the signal that matters for the metric.
- **Cap output length.** Long generations dilute cosine similarity; short ones can't carry enough signal. ~100-150 words is the sweet spot.
- **Akben's post-hoc cosine reranking is cheap and adds points.** If you generate N candidates anyway, picking by cosine to the input is essentially free.

What doesn't generalize:
- The optimal generation strategy here is metric-specific. For a different metric (BLEU, ROUGE, human eval), the answer would change.
- Personality conditioning helped in 2024 but its benefit is shrinking as base models get better at instruction-following style transfer.